In [0]:
# ============================================================
# GOLD DAILY SALES
# Grain: 1 row = 1 day
# Incremental + MERGE + Watermark
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType
)
from delta.tables import DeltaTable


# ============================================================
# 2. CONFIGURATION
# ============================================================

PIPELINE_NAME = "gold_daily_sales"


# ============================================================
# 3. PATHS
# ============================================================

orders_silver_path = (
    "abfss://customer-sales-datalake@"
    "nestlesalesdatalake.dfs.core.windows.net/"
    "silver/orders/"
)

gold_daily_sales_path = (
    "abfss://customer-sales-datalake@"
    "nestlesalesdatalake.dfs.core.windows.net/"
    "gold/daily_sales/"
)

control_path = (
    "abfss://customer-sales-datalake@"
    "nestlesalesdatalake.dfs.core.windows.net/"
    "metadata/pipeline_watermark/"
)


print("================================================")
print("GOLD DAILY SALES PIPELINE")
print("================================================")


# ============================================================
# 4. READ SILVER ORDERS
# ============================================================

orders_df = (
    spark.read
    .format("delta")
    .load(orders_silver_path)
)

print("Silver order records:", orders_df.count())


# ============================================================
# 5. READ / CREATE GOLD WATERMARK
# ============================================================

control_table = DeltaTable.forPath(
    spark,
    control_path
)

control_df = (
    spark.read
    .format("delta")
    .load(control_path)
)


# Check whether Gold Daily watermark exists
watermark_exists = (
    control_df
    .filter(
        F.col("pipeline_name") == PIPELINE_NAME
    )
    .count() > 0
)

print(
    "gold_daily_sales watermark exists:",
    watermark_exists
)


# ============================================================
# 6. CREATE WATERMARK ROW IF MISSING
# ============================================================

watermark_schema = StructType([
    StructField(
        "pipeline_name",
        StringType(),
        False
    ),
    StructField(
        "last_processed_timestamp",
        TimestampType(),
        True
    ),
    StructField(
        "last_run_id",
        StringType(),
        True
    ),
    StructField(
        "updated_at",
        TimestampType(),
        True
    )
])


if not watermark_exists:

    initial_watermark_df = spark.createDataFrame(
        [
            (
                PIPELINE_NAME,
                None,
                None,
                None
            )
        ],
        schema=watermark_schema
    )

    (
        initial_watermark_df
        .write
        .format("delta")
        .mode("append")
        .save(control_path)
    )

    print(
        "Created gold_daily_sales watermark."
    )


# ============================================================
# 7. READ CURRENT GOLD WATERMARK
# ============================================================

control_df = (
    spark.read
    .format("delta")
    .load(control_path)
)

gold_daily_last_processed_timestamp = (
    control_df
    .filter(
        F.col("pipeline_name") == PIPELINE_NAME
    )
    .select(
        "last_processed_timestamp"
    )
    .collect()[0][0]
)

print(
    "Current Gold Daily watermark:",
    gold_daily_last_processed_timestamp
)


# ============================================================
# 8. IDENTIFY INCREMENTAL ORDERS
# ============================================================

if gold_daily_last_processed_timestamp is None:

    print("First Gold Daily run.")
    print("Processing all Silver orders.")

    incremental_orders_df = orders_df

else:

    print(
        "Processing orders after:",
        gold_daily_last_processed_timestamp
    )

    incremental_orders_df = (
        orders_df
        .filter(
            F.col("ingestion_timestamp")
            > gold_daily_last_processed_timestamp
        )
    )


incremental_count = incremental_orders_df.count()

print(
    "Incremental order records:",
    incremental_count
)


# ============================================================
# 9. NO NEW DATA
# ============================================================

if incremental_count == 0:

    print("==============================================")
    print("No new orders found.")
    print("Gold Daily Sales is already up to date.")
    print("Watermark will NOT be updated.")
    print("==============================================")

    dbutils.notebook.exit(
        "SUCCESS: No new records for Gold Daily Sales."
    )


# ============================================================
# 10. FIND AFFECTED DATES
# ============================================================

affected_dates_df = (
    incremental_orders_df
    .select(
        F.to_date(
            F.col("order_timestamp")
        ).alias("sales_date")
    )
    .distinct()
)

print("Affected dates:")
display(
    affected_dates_df
    .orderBy("sales_date")
)


# ============================================================
# 11. READ ALL SILVER ORDERS FOR AFFECTED DATES
# ============================================================
#
# IMPORTANT:
#
# We do NOT aggregate only the incremental records.
#
# If a new order arrives for an existing date,
# we recalculate the COMPLETE day from Silver.
#
# Example:
#
# Oct 6 already has 10,000 orders
# New batch adds 500 orders
#
# Gold must calculate Oct 6 using all 10,500 orders.
#
# ============================================================

affected_orders_df = (
    orders_df
    .withColumn(
        "sales_date",
        F.to_date(
            F.col("order_timestamp")
        )
    )
    .join(
        affected_dates_df,
        on="sales_date",
        how="inner"
    )
)


print(
    "Orders for affected dates:",
    affected_orders_df.count()
)


# ============================================================
# 12. CALCULATE DAILY SALES METRICS
# ============================================================

affected_daily_sales_df = (
    affected_orders_df
    .groupBy("sales_date")
    .agg(
        F.sum(
            "sales_amount"
        ).alias("total_sales"),

        F.countDistinct(
            "order_id"
        ).alias("total_orders"),

        F.sum(
            "quantity"
        ).alias("total_quantity"),

        F.round(
            F.avg("sales_amount"),
            2
        ).alias("average_order_value")
    )
)


print("Daily Gold records to process:")
display(
    affected_daily_sales_df
    .orderBy("sales_date")
)


# ============================================================
# 13. CREATE OR MERGE GOLD TABLE
# ============================================================

gold_table_exists = DeltaTable.isDeltaTable(
    spark,
    gold_daily_sales_path
)


if not gold_table_exists:

    print(
        "Gold Daily Sales table does not exist."
    )

    print(
        "Creating Gold Daily Sales table."
    )

    (
        affected_daily_sales_df
        .write
        .format("delta")
        .mode("overwrite")
        .save(gold_daily_sales_path)
    )

    print(
        "Gold Daily Sales table created."
    )

else:

    print(
        "Gold Daily Sales table exists."
    )

    gold_daily_table = DeltaTable.forPath(
        spark,
        gold_daily_sales_path
    )

    (
        gold_daily_table.alias("target")
        .merge(
            affected_daily_sales_df.alias("source"),
            "target.sales_date = source.sales_date"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(
        "Gold Daily Sales MERGE completed."
    )


# ============================================================
# 14. VERIFY GOLD TABLE
# ============================================================

gold_daily_df = (
    spark.read
    .format("delta")
    .load(gold_daily_sales_path)
)


print(
    "Gold Daily Sales record count:",
    gold_daily_df.count()
)

display(
    gold_daily_df
    .orderBy(
        F.col("sales_date").desc()
    )
    .limit(20)
)


# ============================================================
# 15. VALIDATE DAILY GRAIN
# ============================================================
#
# Grain:
# 1 row = 1 sales_date
#
# Therefore sales_date must be unique.
# ============================================================

duplicate_dates_df = (
    gold_daily_df
    .groupBy("sales_date")
    .count()
    .filter(
        F.col("count") > 1
    )
)


duplicate_date_count = duplicate_dates_df.count()


print(
    "Duplicate sales dates:",
    duplicate_date_count
)


if duplicate_date_count > 0:

    display(
        duplicate_dates_df
        .orderBy("sales_date")
    )

    raise Exception(
        "Gold Daily Sales validation failed: "
        "duplicate sales dates exist."
    )


# ============================================================
# 16. CALCULATE PROCESSED TIMESTAMP
# ============================================================
#
# IMPORTANT:
#
# Use the MAX ingestion_timestamp from the incremental
# Silver records that were actually processed.
#
# Do NOT use current_timestamp() as the watermark.
#
# ============================================================

gold_daily_processed_timestamp = (
    incremental_orders_df
    .agg(
        F.max(
            "ingestion_timestamp"
        ).alias(
            "max_ingestion_timestamp"
        )
    )
    .collect()[0][
        "max_ingestion_timestamp"
    ]
)


print(
    "Gold Daily processed timestamp:",
    gold_daily_processed_timestamp
)


# ============================================================
# 17. CREATE RUN ID
# ============================================================

gold_daily_run_id = (
    f"GOLD_DAILY_SALES_"
    f"{gold_daily_processed_timestamp.strftime('%Y%m%d%H%M%S')}"
)


print(
    "Gold Daily run ID:",
    gold_daily_run_id
)


# ============================================================
# 18. CREATE WATERMARK DATAFRAME
# ============================================================

gold_daily_watermark_df = spark.createDataFrame(
    [
        (
            PIPELINE_NAME,
            gold_daily_processed_timestamp,
            gold_daily_run_id,
            None
        )
    ],
    schema=watermark_schema
).withColumn(
    "updated_at",
    F.current_timestamp()
)


print("Gold Daily watermark to update:")
display(
    gold_daily_watermark_df
)


# ============================================================
# 19. UPDATE SHARED WATERMARK TABLE
# ============================================================

(
    control_table.alias("target")
    .merge(
        gold_daily_watermark_df.alias("source"),
        "target.pipeline_name = source.pipeline_name"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)


print(
    "Gold Daily watermark updated successfully."
)


# ============================================================
# 20. FINAL VERIFICATION
# ============================================================

final_gold_daily_df = (
    spark.read
    .format("delta")
    .load(gold_daily_sales_path)
)


final_watermark_df = (
    spark.read
    .format("delta")
    .load(control_path)
    .filter(
        F.col("pipeline_name") == PIPELINE_NAME
    )
)


print("================================================")
print("GOLD DAILY SALES PIPELINE COMPLETE")
print("================================================")

print(
    "Silver order records:",
    orders_df.count()
)

print(
    "Incremental order records:",
    incremental_count
)

print(
    "Affected dates:",
    affected_dates_df.count()
)

print(
    "Gold Daily records:",
    final_gold_daily_df.count()
)

print(
    "Duplicate sales dates:",
    duplicate_date_count
)

print(
    "Final Gold Daily watermark:"
)

display(
    final_watermark_df
)

print(
    "Final Gold Daily Sales:"
)

display(
    final_gold_daily_df
    .orderBy(
        F.col("sales_date").desc()
    )
    .limit(20)
)